### Tools useage with Calude

#### Write a tool fuction (pyhton function doin a specific task)
#### write a json schema (info of tools to calude)

In [52]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [53]:
# Helper functions
from anthropic.types import Message;

def add_user_message(messages, message):
    user_message = {
        "role": "user",
          "content": message.content if isinstance(message,Message) else message
          }
    messages.append(user_message)


def add_assistant_message(messages, message):
    if isinstance(message,ToolUseBlock):
        assistant_message = {
                "role": "assistant",
                  "content": [{
                        "type" : "tool_use",
                        "id" : message.id,
                        "name": message.name,
                        "input": message.input
                  }]
                  }
    else:
        assistant_message = {
            "role": "assistant",
            "content": message.content if isinstance(message,Message) else message
            }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[],tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        #"temperature": temperature,
        "stop_sequences": stop_sequences,
    }
    if tools:
        params["tools"]=tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message

def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type=="text"]
    )

In [54]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [55]:
from datetime import datetime;
from anthropic.types import ToolParam;
from anthropic.types.tool_use_block import ToolUseBlock;

In [56]:
def get_current_datetime(dateFormat="%Y-%m-%d %H:%M:S"):
    if not dateFormat:
        raise ValueError("Date format cannot be empty");
    return datetime.now().strftime(dateFormat);

get_current_datetime_schema=ToolParam({
  "name": "get_current_datetime",
  "description": "Retrieves the current date and time formatted according to a specified format string. Uses Python's strftime format codes to customize the output. If no format is provided, defaults to 'YYYY-MM-DD HH:MM:SS'. Returns a formatted string representation of the current datetime.",
  "input_schema": {
    "type": "object",
    "properties": {
      "dateFormat": {
        "type": "string",
        "description": "A Python strftime format string that specifies how to format the returned datetime (e.g., '%Y-%m-%d %H:%M:%S' for '2026-09-20 14:30:45', '%Y/%m/%d' for '2026/09/20'). Defaults to '%Y-%m-%d %H:%M:%S'. Cannot be an empty string.",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": []
  }
})

In [57]:
messageList=[]
messageList.append({
    "role" : "user",
    "content": "what is the exact time , formatted as HH:MM:SS ?"
})


response=client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messageList,
    tools=[get_current_datetime_schema],
    
)






In [58]:
messageList.append({
    "role": "assistant",
    "content": [
        {
            "type" : "tool_use",
            "id" : response.content[0].id,
            "name": response.content[0].name,
            "input": response.content[0].input
        }
    ]
})

In [59]:
result=get_current_datetime(**response.content[0].input)

In [60]:
messageList.append({
    "role": "user",
    "content" : [{
        "type" :"tool_result",
        "tool_use_id" : response.content[0].id,
        "content" : result,
        "is_error": False
    }]
})

In [61]:
messageList

[{'role': 'user',
  'content': 'what is the exact time , formatted as HH:MM:SS ?'},
 {'role': 'assistant',
  'content': [{'type': 'tool_use',
    'id': 'toolu_01521Qc1KBw9SKfY2FTbgjTu',
    'name': 'get_current_datetime',
    'input': {'dateFormat': '%H:%M:%S'}}]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01521Qc1KBw9SKfY2FTbgjTu',
    'content': '10:51:46',
    'is_error': False}]}]

In [62]:
anotherResponse=client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messageList,
    tools=[get_current_datetime_schema]
)

In [63]:
anotherResponse.content[0].text

'The exact time is **10:51:46**.'

#### what is 103 days from today ?

In [64]:
anotherMessageList=[]
add_user_message(anotherMessageList,"What is the current time in HH:MM:SS format ?")

In [65]:
newResponse=client.messages.create(
    model=model,
    max_tokens=1000,
    messages=anotherMessageList,
    tools=[get_current_datetime_schema]
)

In [66]:
add_assistant_message(anotherMessageList,newResponse);

In [67]:
anotherMessageList

[{'role': 'user', 'content': 'What is the current time in HH:MM:SS format ?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01NxqQjSWHBZEL2LU5NPA4kr', caller=DirectCaller(type='direct'), input={'dateFormat': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]}]

In [68]:
newResponse

Message(id='msg_011CfU45LedhuCG7VCHABtrC', container=None, content=[ToolUseBlock(id='toolu_01NxqQjSWHBZEL2LU5NPA4kr', caller=DirectCaller(type='direct'), input={'dateFormat': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=733, output_tokens=62, output_tokens_details=None, server_tool_use=None, service_tier='standard'), diagnostics=None)

In [69]:
def run_tool(tool_name,tool_input):
    if tool_name=="get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name=="add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name=="set_reminder":
        return set_reminder(**tool_input)

def run_tools(message):
    #tool_requests=[block for block in message.content if block.type=="tool_use"]
    tool_result_block=[]
    #for tool in tool_requests:
    tool_output=run_tool(message.name,message.input)
    print(tool_output)
    try:
        tool_request_block={
            "type" :"tool_result",
            "tool_use_id" : message.id,
            "content" : tool_output,
            "is_error": False
        }
    except Exception as e:
        tool_request_block={
            "type" :"tool_result",
            "tool_use_id" : message.id,
            "content" : f"Error: {e}",
            "is_error": True
            }
        
    tool_result_block.append(tool_request_block)
    return tool_result_block

In [70]:
def run_conversation(messageList):
    while True:
        response=chat(messageList,tools=[get_current_datetime_schema,
                                        add_duration_to_datetime_schema,
                                        set_reminder_schema
                                         ])
        #add_assistant_message(messageList,response)
        
        if response.stop_reason !="tool_use":
            break
        for tool_call in response.content:
            if tool_call.type=='tool_use':
                #print(tool_call)
                tool_results=run_tools(tool_call)
                add_assistant_message(messageList,tool_call)
                add_user_message(messageList,tool_results)
    return response

In [71]:
mymessageList=[]
add_user_message(mymessageList,"set aremider for my doctors appointment . Its 7 days after 29-sep-2026")

In [72]:
ans=run_conversation(mymessageList)

Tuesday, October 06, 2026 12:00:00 AM
----
Setting the following reminder for 2026-10-06T12:00:00:
Doctor's appointment
----
None


In [74]:
print(ans.content[0].text)

Perfect! I've set a reminder for your doctor's appointment on **Tuesday, October 6, 2026 at 12:00 AM** (7 days after September 29, 2026). You'll receive a notification at that time.


In [75]:
mymessageList

[{'role': 'user',
  'content': 'set aremider for my doctors appointment . Its 7 days after 29-sep-2026'},
 {'role': 'assistant',
  'content': [{'type': 'tool_use',
    'id': 'toolu_01GNvv1vErpXZcX1Yoxbc434',
    'name': 'add_duration_to_datetime',
    'input': {'datetime_str': '2026-09-29',
     'input_format': '%Y-%m-%d',
     'duration': 7,
     'unit': 'days'}}]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01GNvv1vErpXZcX1Yoxbc434',
    'content': 'Tuesday, October 06, 2026 12:00:00 AM',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [{'type': 'tool_use',
    'id': 'toolu_01GTxeKCXx5nFjPNu97FjchG',
    'name': 'set_reminder',
    'input': {'content': "Doctor's appointment",
     'timestamp': '2026-10-06T12:00:00'}}]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01GTxeKCXx5nFjPNu97FjchG',
    'content': None,
    'is_error': False}]}]